In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


## Loading the Analysis-Ready Dataset

This step loads the processed NHANES analysis-ready dataset from a Parquet file. This dataset was created after merging the selected NHANES cycles, harmonizing variables across cycles, constructing the diabetes outcome variable, applying the population criteria, selecting the required predictors, and renaming the columns to interpretable feature names.

After loading the dataset, the code prints the available columns to verify that the expected variables are present. It also displays the class distribution of the `diabetes` outcome variable, both as raw counts and as proportions. This is used to confirm the degree of class imbalance before applying preprocessing, train-test splitting, and model training.

In [5]:
df = pd.read_parquet("../data/raw/nhanes_diabetes_raw.parquet")
print(df.columns)
print(df["diabetes"].value_counts())
print(df["diabetes"].value_counts(normalize=True))


Index(['participant_id', 'cycle', 'diabetes', 'age', 'sex', 'race_ethnicity',
       'education_level', 'income_poverty_ratio', 'bmi', 'waist_circumference',
       'mean_systolic_bp', 'mean_diastolic_bp', 'ever_smoked_100_cigarettes',
       'average_alcoholic_drinks_per_day', 'vigorous_work_activity',
       'moderate_work_activity', 'walk_or_bicycle_transport',
       'vigorous_recreational_activity', 'moderate_recreational_activity',
       'sleep_hours', 'total_cholesterol_mg_dl', 'hdl_cholesterol_mg_dl',
       'triglycerides_mg_dl', 'ldl_cholesterol_mg_dl', 'energy_kcal',
       'protein_g', 'carbohydrate_g', 'total_sugar_g', 'fiber_g',
       'total_fat_g', 'cholesterol_mg'],
      dtype='object')
diabetes
0    21092
1     4909
Name: count, dtype: Int64
diabetes
0    0.8112
1    0.1888
Name: proportion, dtype: Float64


## Splitting the Dataset into Training and Test Sets

This step separates the analysis-ready dataset into predictor variables (`X`) and the target variable (`y`). The participant identifier, survey cycle, and diabetes outcome are removed from `X`, because they should not be used as model predictors. The `diabetes` column is stored separately as the target variable.

The dataset is then split into a training set and a test set using an 80/20 split. The `stratify=y` argument ensures that the proportion of diabetic and non-diabetic participants remains approximately the same in both the training and test sets. This is important because the diabetes outcome is imbalanced.

The training set will be used for preprocessing, imbalance handling, and model training. The test set will remain untouched until final evaluation, so that model performance is measured on unseen data.

In [6]:
X = df.drop(columns=["participant_id", "cycle", "diabetes"])
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train = X_train.reset_index(drop=True).copy()
X_test = X_test.reset_index(drop=True).copy()
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


(20800, 28) (5201, 28)
diabetes
0    0.811202
1    0.188798
Name: proportion, dtype: Float64
diabetes
0    0.81119
1    0.18881
Name: proportion, dtype: Float64


## Cleaning alcohol consumption values

The alcohol consumption variable contains special NHANES response codes such as `777` for refused answers and `999` for "don't know". These values do not represent valid alcohol intake and are therefore converted to missing values. Extremely high values above 30 drinks per day are also treated as invalid outliers. 

Because this variable contains many missing values, a separate missingness indicator is created before filling missing alcohol values with 0. This prevents median imputation from assigning artificial average alcohol consumption to participants without valid alcohol information.

In [7]:
alcohol_col = "average_alcoholic_drinks_per_day"

X_train[alcohol_col] = X_train[alcohol_col].replace({
    777: np.nan,
    999: np.nan
})

X_test[alcohol_col] = X_test[alcohol_col].replace({
    777: np.nan,
    999: np.nan
})

X_train.loc[X_train[alcohol_col] > 30, alcohol_col] = np.nan
X_test.loc[X_test[alcohol_col] > 30, alcohol_col] = np.nan

X_train["alcohol_info_missing"] = X_train[alcohol_col].isna().astype(int)
X_test["alcohol_info_missing"] = X_test[alcohol_col].isna().astype(int)

X_train[alcohol_col] = X_train[alcohol_col].fillna(0)
X_test[alcohol_col] = X_test[alcohol_col].fillna(0)

## Recoding binary yes/no variables

Several NHANES questionnaire variables are coded as `1 = yes` and `2 = no`, while `7` and `9` indicate refused or unknown responses. These variables are recoded to binary format, where `1` represents yes and `0` represents no. Refused and unknown responses are converted to missing values so they can be handled consistently during imputation.

In [8]:
yes_no_cols = [
    "ever_smoked_100_cigarettes",
    "vigorous_work_activity",
    "moderate_work_activity",
    "walk_or_bicycle_transport",
    "vigorous_recreational_activity",
    "moderate_recreational_activity",
]

for col in yes_no_cols:
    X_train[col] = X_train[col].replace({
        1: 1,
        2: 0,
        7: np.nan,
        9: np.nan
    })

    X_test[col] = X_test[col].replace({
        1: 1,
        2: 0,
        7: np.nan,
        9: np.nan
    })

## Handling special codes in education and sleep variables

The education and sleep variables may contain special response codes that do not represent valid values. For example, `7`, `9`, `77`, and `99` are used for responses such as refused or don't know. These codes are converted to missing values before imputation.

In [9]:
X_train["education_level"] = X_train["education_level"].replace({
    7: np.nan,
    9: np.nan
})

X_test["education_level"] = X_test["education_level"].replace({
    7: np.nan,
    9: np.nan
})

In [10]:
X_train["sleep_hours"] = X_train["sleep_hours"].replace({
    77: np.nan,
    99: np.nan
})

X_test["sleep_hours"] = X_test["sleep_hours"].replace({
    77: np.nan,
    99: np.nan
})

## Missing value imputation

Missing values are imputed separately for numeric, binary, and categorical features. Numeric features are filled using the median value from the training set, while binary and categorical features are filled using the most frequent value from the training set. The same imputation values learned from the training data are then applied to the test set to prevent data leakage.

In [11]:
numeric_features = [
    "age",
    "income_poverty_ratio",
    "bmi",
    "waist_circumference",
    "mean_systolic_bp",
    "mean_diastolic_bp",
    "sleep_hours",
    "average_alcoholic_drinks_per_day",
    "energy_kcal",
    "protein_g",
    "carbohydrate_g",
    "total_sugar_g",
    "fiber_g",
    "total_fat_g",
    "cholesterol_mg",
    "total_cholesterol_mg_dl",
    "hdl_cholesterol_mg_dl",
    "triglycerides_mg_dl",
    "ldl_cholesterol_mg_dl",
]

binary_features = [
    "ever_smoked_100_cigarettes",
    "vigorous_work_activity",
    "moderate_work_activity",
    "walk_or_bicycle_transport",
    "vigorous_recreational_activity",
    "moderate_recreational_activity",
    "alcohol_info_missing",
]

categorical_features = [
    "sex",
    "race_ethnicity",
    "education_level",
]

numeric_medians = X_train[numeric_features].median()

X_train[numeric_features] = X_train[numeric_features].fillna(numeric_medians)
X_test[numeric_features] = X_test[numeric_features].fillna(numeric_medians)

binary_modes = X_train[binary_features].mode().iloc[0]

X_train[binary_features] = X_train[binary_features].fillna(binary_modes)
X_test[binary_features] = X_test[binary_features].fillna(binary_modes)

categorical_modes = X_train[categorical_features].mode().iloc[0]

X_train[categorical_features] = X_train[categorical_features].fillna(categorical_modes)
X_test[categorical_features] = X_test[categorical_features].fillna(categorical_modes)

# Verify no missing values remain before encoding
all_imputed = numeric_features + binary_features + categorical_features
assert X_train[all_imputed].isna().sum().sum() == 0, "Missing values in train after imputation!"
assert X_test[all_imputed].isna().sum().sum() == 0, "Missing values in test after imputation!"
print("[OK] No missing values after imputation.")


[OK] No missing values after imputation.


## Encoding categorical variables

Categorical variables are converted into numeric format using one-hot encoding. Only multi-category variables such as sex, race/ethnicity, and education level are encoded. Binary variables are kept as 0/1 features and are not one-hot encoded. After encoding, the training and test sets are aligned to ensure that both datasets contain the same columns.

In [12]:
X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_features,
    drop_first=True
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_features,
    drop_first=True
)

X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

# Verify all numeric features still exist after encoding
assert all(c in X_train_encoded.columns for c in numeric_features), "Numeric column missing after encoding!"
print(f"[OK] Encoded shape: {X_train_encoded.shape}")


[OK] Encoded shape: (20800, 36)


## Scaling Numerical Features

This step standardizes the numerical predictor variables using `StandardScaler`. Standardization transforms each numerical feature so that it has a mean of approximately 0 and a standard deviation of 1.

The scaler is fitted only on the training set using `fit_transform()`. The same fitted scaler is then applied to the test set using `transform()`. This prevents data leakage, because information from the test set is not used to calculate the scaling parameters.

Scaling is especially important for models such as Logistic Regression, because these models are sensitive to differences in feature scale. Although tree-based models such as Decision Tree, Random Forest, and XGBoost do not strictly require scaling, the same scaled feature matrix is used to keep the preprocessing setup consistent across models.

In [13]:
scaler = StandardScaler()

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train_scaled[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test_scaled[numeric_features])

## Defining the Final Preprocessed Feature Sets

This step assigns the fully preprocessed training and test sets to  and . These datasets have already gone through missing-value imputation, categorical encoding, and numerical feature scaling.

The final feature sets will be used for both class-imbalance strategies. In Strategy A, SMOTE is applied to  inside the cross-validation pipeline. In Strategy B, random undersampling is applied inside the cross-validation pipeline. In both strategies, model performance is evaluated on the same untouched .


In [14]:
X_train_final = X_train_scaled
X_test_final = X_test_scaled

print("Missing train:", X_train_final.isna().sum().sum())
print("Missing test:", X_test_final.isna().sum().sum())
print("Train shape:", X_train_final.shape)
print("Test shape:", X_test_final.shape)
print("Same columns:", list(X_train_final.columns) == list(X_test_final.columns))

import os
os.makedirs("../data/processed", exist_ok=True)

X_train_final.to_parquet("../data/processed/X_train_final.parquet", index=False)
X_test_final.to_parquet("../data/processed/X_test_final.parquet", index=False)
y_train.to_frame(name="diabetes").to_parquet("../data/processed/y_train.parquet", index=False)
y_test.to_frame(name="diabetes").to_parquet("../data/processed/y_test.parquet", index=False)

print("[OK] All files saved to ../data/processed/")


Missing train: 0
Missing test: 0
Train shape: (20800, 36)
Test shape: (5201, 36)
Same columns: True
[OK] All files saved to ../data/processed/
